<a href="https://colab.research.google.com/github/JorgeZorrilla/Crash-GeoNN/blob/main/TrainingModel_iteration_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:
- Meter velocidades
- Meter aceleraciones
- Diferenciar los distintos solidos
- Meter features estaticos.
- Meter los del validation a k steps para tener mejores predicciones en el futuro
- Probar lo del CLAMP_BC_IN_ROLLOUT

## Configuration

In [149]:
# Configuration parameters
SEED_NUMBER = 42
MIN_T = 1
STEP = 2 # To select a smaller number of attributes from the database
LAM_BC = 1e-2
LAM_SMOOTH = 1e-3
CLAMP_BC_IN_ROLLOUT = False
PATIENCE = 15
MAX_EPOCHS = 200
N_LAYERS= 3
HIDDEN= 128


## Install dependencies

In [150]:
# Colab setup: install PyTorch Geometric wheels matching your Torch/CUDA
import torch, sys, os, platform, subprocess, textwrap
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# This magic line pulls the right wheels for your torch+cuda combo
torch_ver = torch.__version__.split('+')[0]
cuda_tag = (torch.version.cuda or 'cpu').replace('.', '')
index_url = f"https://data.pyg.org/whl/torch-{torch_ver}%2B{cuda_tag}.html"

!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric \
  -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

!pip install pyvista imageio-ffmpeg

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)



Torch: 2.8.0+cu126 | CUDA: 12.6
Device: cuda


Mount drive

In [151]:
from google.colab import drive
drive.mount('/content/drive')  # autoriza y usa rutas como '/content/drive/MyDrive/...'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import dependencies

In [152]:
import os, math, random, numpy as np, time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Dict, List, Tuple, Sequence
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GraphSAGE
from tqdm.auto import tqdm

Utilities

In [153]:
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    try: torch.set_float32_matmul_precision("high")
    except: pass

def worker_init_fn(worker_id):
    seed = torch.initial_seed() % 2**31
    np.random.seed(seed + worker_id); random.seed(seed + worker_id)

def check_sim(steps: List[Data], sid: int, max_print_edges=5):
    assert isinstance(steps, (list, tuple)) and len(steps) >= 1, f"[sim {sid}] bad list"
    N = steps[0].x.shape[0]
    E = steps[0].edge_index.shape[1]
    pos0 = getattr(steps[0], 'pos0', None)
    edge_index0 = steps[0].edge_index
    issues = []
    for t, g in enumerate(steps):
        if not isinstance(g, Data): issues.append(f"step {t} not Data"); continue
        if g.x.dim()!=2 or g.y.dim()!=2: issues.append(f"step {t} x/y dim !=2")
        if g.x.shape[0]!=N or g.y.shape[0]!=N: issues.append(f"step {t} N mismatch")
        if g.edge_index.shape[0]!=2 or g.edge_index.shape[1]!=E: issues.append(f"step {t} ei shape mismatch")
        if not torch.equal(g.edge_index, edge_index0): issues.append(f"step {t} ei differs")
        if int(g.edge_index.max()) >= N: issues.append(f"step {t} ei out of range")
        if not torch.isfinite(g.x).all() or not torch.isfinite(g.y).all(): issues.append(f"step {t} NaN/Inf in x/y")
        if hasattr(g, "edge_attr"):
            if not torch.isfinite(g.edge_attr).all(): issues.append(f"step {t} NaN/Inf in edge_attr")
    ei = edge_index0.t().tolist()
    undirected = all(([j,i] in ei) for i,j in ei[:max_print_edges])
    unique_pairs = set(tuple(sorted(e)) for e in ei)
    dup = (len(unique_pairs) * 2 != len(ei))
    print(f"[sim {sid}] N={N} E={E} undirected? {undirected} duplicates? {dup}")
    if issues: print("  Issues:", "; ".join(issues))

def transform_edge_attr(edge_attr: torch.Tensor, edge_scaler):
    if edge_attr is None or edge_scaler is None:
        return edge_attr
    em, es = edge_scaler
    return (edge_attr - em) / es

def drop_features_db(db: List[List[Data]], drop_idx_x: List[int], drop_idx_y: List[int]):
    """
    Elimina atributos (columnas) de x (y opcionalmente de y) en TODA la base de datos.

    Args:
        db: List[List[Data]]  -> base de datos completa
        drop_idx: lista de índices de columnas a eliminar
    """
    if not drop_idx_x and not drop_idx_y:
        return db  # nada que hacer

    drop_idx_x = sorted(set(drop_idx_x))
    drop_idx_y = sorted(set(drop_idx_y))

    for sim in db:
        for g in sim:
            # --- X ---
            if hasattr(g, "x") and g.x is not None:
                keep_x = [i for i in range(g.x.size(1)) if i not in drop_idx_x]
                g.x = g.x[:, keep_x]

            # --- Y (opcional) ---
            if hasattr(g, "y") and g.y is not None:
              keep_y = [i for i in range(g.y.size(1)) if i not in drop_idx_y]
              g.y = g.y[:, keep_y]


    return db



def tensor3d_to_dfs(arr3d, feature_names=None):
    """
    arr3d: (T, N, C) en numpy o torch
    feature_names: lista de nombres de longitud C (opcional)
    """
    # -> numpy
    if isinstance(arr3d, torch.Tensor):
        A = arr3d.detach().cpu().numpy()
    else:
        A = np.asarray(arr3d)
    T, N, C = A.shape

    # Nombres de columnas
    cols = feature_names if feature_names is not None else [f"f{i}" for i in range(C)]

    # ---- Wide: index=(t,node), columns=features ----
    idx = pd.MultiIndex.from_product([range(T), range(N)], names=["t", "node"])
    df_wide = pd.DataFrame(A.reshape(T*N, C), index=idx, columns=cols)

    # ---- Long: tidy ----
    t_idx   = np.repeat(np.arange(T), N*C)
    node_idx= np.tile(np.repeat(np.arange(N), C), T)
    feat_idx= np.tile(np.arange(C), T*N)
    feat    = np.array(cols)[feat_idx]
    values  = A.reshape(-1)
    df_long = pd.DataFrame({"t": t_idx, "node": node_idx, "feature": feat, "value": values})

    return df_long, df_wide

In [154]:
set_seed(SEED_NUMBER)

Load Database

In [155]:
def load_database(path_pt: str) -> List[List[Data]]:
    # print("Loading DB from:", path_pt)
    db = torch.load(path_pt, map_location="cpu", weights_only=False)
    return db

def load_database_dir(dir_path: str, step = 1) -> List[List[Data]]:
    print("Loading DB from:", dir_path)
    db = []
    graphs = os.listdir(dir_path)
    if graphs:
      print(f"Found {len(graphs)} graphs")
      for i in tqdm(range(0, len(graphs), step)):
        path = os.path.join(dir_path, graphs[i])
        db.append(load_database(path))
      for sid, steps in enumerate(db[:5]): check_sim(steps, sid)
      lens = [len(s) for s in db]
    return db
# def load_database_dir(path_pt: str, step = 1) -> List[List[Data]]:
#     print("Loading DB from:", path_pt)
#     db = []
#     graphs = os.listdir(path_pt)
#     if graphs:
#       print(f"Found {len(graphs)} graphs")
#       for i in range(0, len(graphs), step):
#         print(f"Loading graph {graphs[i]}")
#         path = os.path.join(path_pt, graphs[i])
#         db.append(torch.load(path, map_location="cpu", weights_only=False))
#       assert isinstance(db, (list, tuple)) and all(isinstance(sim, (list, tuple)) for sim in db)
#       for sid, steps in enumerate(db[:5]): check_sim(steps, sid)
#       lens = [len(s) for s in db]
#       print(f"T-1 per sim (min/mean/max): {min(lens)}/{sum(lens)/len(lens):.1f}/{max(lens)}")
#       print(f"Loaded ${len(db)} graphs!")
#     return db

def split_simulations(all_sim_ids, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = np.random.default_rng(seed); ids = np.array(all_sim_ids); rng.shuffle(ids)
    n = len(ids); n_tr = int(n*train_ratio); n_va = int(n*val_ratio)
    return ids[:n_tr].tolist(), ids[n_tr:n_tr+n_va].tolist(), ids[n_tr+n_va:].tolist()

def build_split_from_db(db: List[List[Data]], sim_ids: List[int], min_time_step: int = 0):
    graphs, sim_static = [], {}
    for sid in sim_ids:
        steps_all = db[sid]
        assert len(steps_all) >= 1, f"Simulation {sid} empty."

        # Si no hay suficientes pasos, saltamos la simulación
        if len(steps_all) <= min_time_step:
            print(f"[WARN] sim {sid} skipped: len(steps)={len(steps_all)} <= min_t={min_time_step}")
            continue

        # Filtrado por timestep
        steps = steps_all[min_time_step:]                        # Data_t(min_time_step) .. Data_t(T-2)
        edge_index = steps[0].edge_index
        pos0 = getattr(steps[0], 'pos0', None)
        simulation_id = getattr(steps[0], 'simulation_id', None)
        bc_mask = getattr(steps[0], 'bc_mask', None)
        rigid_mask = getattr(steps[0], 'rigid_mask', None)
        timestep_index = getattr(steps[0], 't_idx', None)
        # fixed_idx = getattr(steps[0], 'fixed_idx', None)
        edge_attr = getattr(steps[0], 'edge_attr', None)

        # K = len(steps) = (T-1 - min_time_step)
        # Estados efectivos: ΔX_{min_time_step} .. ΔX_T  -> T_eff = K + 1
        T_eff = len(steps) + 1

        # Ground-truth a partir de min_time_step: ΔX_{min_time_step+1 .. T}
        y_real = torch.stack([d.y for d in steps], dim=0)  # (T_eff-1, N, N_features)

        # Estado inicial para rollout: ΔX_{min_t} (ojo: sin normalizar)
        x0 = steps_all[min_time_step].x.detach().clone()

        # Añadimos los Data filtrados al conjunto de entrenamiento/val/test
        graphs.extend(steps)

        sim_static[sid] = {
            'simulation_id' : simulation_id,
            'bc_mask' : bc_mask,
            'rigid_mask' : rigid_mask,
            'timestep_index' : timestep_index,
            # 'fixed_idx' : fixed_idx,
            'edge_index': edge_index,
            'edge_attr' : edge_attr,     # OJO: aún sin escalar aquí
            'pos0': pos0,
            'T_eff': T_eff, # Number of effective timesteps
            'y_real': y_real,
            'x0': x0           # punto de partida del rollout
        }

    return graphs, sim_static

In [156]:
# BBDD parameters
INPUT_DIR="/content/drive/MyDrive/CrashGeoNN/graphs_iteration_3/"
# Attributes [Delta_x, Delta_y, Delta_z, V_x, V_y, V_z, A_z, A_y, A_z, bc_mask, rigid_mask]
# N_ FEATURES = 11
# Iteration 1: Only displacements
# X_DISCARD_INDEX = [3,4,5,6,7,8] # We start only with the coords and the static masks
# X_DYNAMIC_INDEX = [0,1,2]
# X_STATIC_INDEX = [3, 4] # After removing the previous index bc_mask and rigid_mask remain
# Y_DISCARD_INDEX = [3,4,5,6,7,8] # We start only with the coords and the static masks
# Y_DYNAMIC_INDEX = [0,1,2]
# Y_STATIC_INDEX = [] # No static attributes here
# Iteration 2: Displacements + velocties
X_DISCARD_INDEX = [6,7,8] # We discard accelerations
X_DYNAMIC_INDEX = [0,1,2,3,4,5]
X_STATIC_INDEX = [6,7] # After removing the previous index bc_mask and rigid_mask remain
Y_DISCARD_INDEX = [6,7,8] # We discard accelerations
Y_DYNAMIC_INDEX = [0,1,2,3,4,5]
Y_STATIC_INDEX = [] # No static attributes here

In [157]:
# === Cambia esta ruta a tu .pt (Drive o local) ===
DB_PATH = INPUT_DIR  # p.ej.: "/content/drive/MyDrive/Crash-GeoNN/GRAPHS.pt"
simulations = load_database_dir(DB_PATH, STEP)


Loading DB from: /content/drive/MyDrive/CrashGeoNN/graphs_iteration_3/
Found 93 graphs


  0%|          | 0/47 [00:00<?, ?it/s]

[sim 0] N=3544 E=28108 undirected? True duplicates? False
[sim 1] N=3499 E=27772 undirected? True duplicates? False
[sim 2] N=3546 E=28152 undirected? True duplicates? False
[sim 3] N=3555 E=28196 undirected? True duplicates? False
[sim 4] N=3550 E=28168 undirected? True duplicates? False


In [158]:
print(f"Number of features before: {simulations[0][0].x.shape[1]}")
simulations = drop_features_db(simulations,X_DISCARD_INDEX, Y_DISCARD_INDEX)
INPUT_FEATURES = simulations[0][0].x.shape[1]
OUTPUT_FEATURES = simulations[0][0].y.shape[1]
print(f"Final number of INPUT features: {INPUT_FEATURES}. Dynamic: {len(X_DYNAMIC_INDEX)}. Static: {len(X_STATIC_INDEX)}")
assert INPUT_FEATURES == (len(X_DYNAMIC_INDEX) + len(X_STATIC_INDEX))
print(f"Final number of OUTPUT features: {OUTPUT_FEATURES}. Dynamic: {len(Y_DYNAMIC_INDEX)}. Static: {len(Y_STATIC_INDEX)}")
assert OUTPUT_FEATURES == (len(Y_DYNAMIC_INDEX) + len(Y_STATIC_INDEX))

assert len(X_DYNAMIC_INDEX) == len(Y_DYNAMIC_INDEX) # Needed for the rollout



all_ids = list(range(len(simulations)))
train_ids, val_ids, test_ids = split_simulations(all_ids, train_ratio=0.7, val_ratio=0.15, seed=42)

Number of features before: 11
Final number of INPUT features: 8. Dynamic: 6. Static: 2
Final number of OUTPUT features: 6. Dynamic: 6. Static: 0


In [159]:
print("Building splits...")
train_graphs, train_static = build_split_from_db(simulations, train_ids, min_time_step=MIN_T)
val_graphs,   val_static   = build_split_from_db(simulations, val_ids, min_time_step=MIN_T)
test_graphs,  test_static  = build_split_from_db(simulations, test_ids, min_time_step=MIN_T)


Building splits...


## Normalization

In [160]:
def fit_scaler(graphs: List[Data],
               affected_index_x: Sequence[int],
               affected_index_y: Sequence[int]):

    # Complete dimensions
    len_x = graphs[0].x.size(-1)
    len_y = graphs[0].y.size(-1)

    # Scaler X: only the affected columns
    Xc = torch.cat([g.x[:, affected_index_x] for g in graphs], dim=0)
    xm_c = Xc.mean(0, keepdim=True)
    xs_c = Xc.std(0, keepdim=True).clamp_min(1e-8)

    # We include average mu = 0 and std = 1 to the non affected attributes
    xm = torch.zeros(1, len_x, dtype=xm_c.dtype, device=xm_c.device)
    xs = torch.ones(1,  len_x, dtype=xs_c.dtype, device=xs_c.device)
    xm[:, affected_index_x] = xm_c # Modify affected attributes with average
    xs[:, affected_index_x] = xs_c # Modify affected attributes with average

    # Scaler Y: only the affected columns
    Yc = torch.cat([g.y[:, affected_index_y] for g in graphs], dim=0)
    ym_c = Yc.mean(0, keepdim=True)
    ys_c = Yc.std(0, keepdim=True).clamp_min(1e-8)

    # We include average mu = 0 and std = 1 to the non affected attributes
    ym = torch.zeros(1, len_y, dtype=ym_c.dtype, device=ym_c.device)
    ys = torch.ones(1,  len_y, dtype=ys_c.dtype, device=ys_c.device)
    ym[:, affected_index_y] = ym_c # Modify affected attributes with average
    ys[:, affected_index_y] = ys_c # Modify affected attributes with average

    return (xm, xs), (ym, ys)


def apply_scaler(graphs: List[Data], x_scaler, y_scaler):
    xm, xs = x_scaler
    ym, ys = y_scaler
    for g in graphs:
        dev = g.x.device
        g.x = (g.x - xm.to(dev)) / xs.to(dev)
        g.y = (g.y - ym.to(dev)) / ys.to(dev)

def fit_edge_attr_scaler(graphs: List[Data]):
    E_list = [g.edge_attr for g in graphs if hasattr(g, "edge_attr") and g.edge_attr is not None]
    if not E_list: return None
    E = torch.cat(E_list, dim=0)
    em, es = E.mean(0, keepdim=True), E.std(0, keepdim=True).clamp_min(1e-8)
    return (em, es)

def apply_edge_attr_scaler(graphs: List[Data], scaler):
    if scaler is None: return
    em, es = scaler
    for g in graphs:
        if hasattr(g, "edge_attr") and g.edge_attr is not None:
            g.edge_attr = (g.edge_attr - em) / es

def denorm(x_norm, mean, std):
    return x_norm * std.to(x_norm) + mean.to(x_norm)

def fit_delta_scaler(graphs, dyn_idx_x, dyn_idx_y):
    """
    graphs: lista de Data (TRAIN) en físico (AÚN SIN normalizar)
    dyn_idx_x: índices dinámicos en x (los que se actualizan con el modelo)
    dyn_idx_y: índices dinámicos en y (deberían corresponder 1:1 con dyn_idx_x)
    """
    with torch.no_grad():
        deltas = []
        for g in graphs:
            x_dyn = g.x[:, dyn_idx_x]     # (N, Ddyn)
            y_dyn = g.y[:, dyn_idx_y]     # (N, Ddyn)
            deltas.append(y_dyn - x_dyn)  # Δ_phys
        D = torch.cat(deltas, dim=0)      # (TotalN, Ddyn)

        dm = D.mean(0, keepdim=True)
        ds = D.std(0, keepdim=True).clamp_min(1e-8)
    return (dm, ds)

def denorm_with_scaler(x_norm, scaler):
    m, s = scaler
    return x_norm * s.to(x_norm) + m.to(x_norm)

def norm_with_scaler(x_phys, scaler):
    m, s = scaler
    return (x_phys - m.to(x_phys)) / s.to(x_phys)


In [161]:
print("Fitting scalers on TRAIN...")

x_scaler, y_scaler = fit_scaler(train_graphs, X_DYNAMIC_INDEX, Y_DYNAMIC_INDEX)
delta_scaler = fit_delta_scaler(train_graphs, X_DYNAMIC_INDEX, Y_DYNAMIC_INDEX)
apply_scaler(train_graphs, x_scaler, y_scaler)
apply_scaler(val_graphs,   x_scaler, y_scaler)
apply_scaler(test_graphs,  x_scaler, y_scaler)

edge_scaler = fit_edge_attr_scaler(train_graphs)
apply_edge_attr_scaler(train_graphs, edge_scaler)
apply_edge_attr_scaler(val_graphs,   edge_scaler)
apply_edge_attr_scaler(test_graphs,  edge_scaler)

Fitting scalers on TRAIN...


Models

In [162]:
from torch_geometric.nn import GINEConv, BatchNorm, LayerNorm, GraphNorm

class ImpactGNN(nn.Module):
    def __init__(self, in_ch=3, hidden=128, out_ch=3, layers=3):
        super().__init__()
        self.gnn = GraphSAGE(in_channels=in_ch, hidden_channels=hidden, num_layers=layers)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, out_ch))
    def forward(self, x, edge_index, edge_attr=None):
        h = self.gnn(x, edge_index)
        return self.head(h)

class ImpactGNN_Edge(nn.Module):
    def __init__(self, in_ch=3, edge_attr_dim=4, hidden=128, out_ch=3, layers=3, dropout=0.1):
        super().__init__()
        convs, norms = [], []
        for l in range(layers):
            mlp = nn.Sequential(
                nn.Linear(hidden if l>0 else in_ch, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            )
            convs.append(GINEConv(mlp, edge_dim=edge_attr_dim))
            norms.append(BatchNorm(hidden))
        self.convs = nn.ModuleList(convs)
        self.norms = nn.ModuleList(norms)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_ch))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = x
        for conv, bn in zip(self.convs, self.norms):
            h = conv(h, edge_index, edge_attr)
            h = bn(h); h = F.relu(h); h = self.dropout(h)
        return self.head(h)

In [163]:
print("Creating loaders...")
loader_kwargs = dict(batch_size=16, shuffle=True, pin_memory=(device=='cuda'),
                      num_workers=0, worker_init_fn=worker_init_fn, persistent_workers=False)
train_loader = DataLoader(train_graphs, **loader_kwargs)
val_loader   = DataLoader(val_graphs,   **{**loader_kwargs, "shuffle": False})
test_loader  = DataLoader(test_graphs,  **{**loader_kwargs, "shuffle": False})

Creating loaders...


Training

In [164]:
def smooth_edge_penalty(pred, target, edge_index, lam=1e-3,
                        bc_mask=None, solid_id=None):
    """Match the gradient of the edge pred vs target, but ignores
    the edges that have nodes in the BC or belong to different solids."""
    src, dst = edge_index
    diff = (pred[src] - pred[dst]) - (target[src] - target[dst])  # [E, C]

    if bc_mask is not None:
        free_edge = ((bc_mask[src] == 0) & (bc_mask[dst] == 0)).unsqueeze(-1)  # [E,1]
        diff = diff * free_edge

    if solid_id is not None:
        same_solid = (solid_id[src] == solid_id[dst]).unsqueeze(-1)  # [E,1]
        diff = diff * same_solid

    return lam * diff.pow(2).mean()

def masked_mse(pred, target, mask_free):
    # mask_free: True en nodos libres
    if mask_free is None: return F.mse_loss(pred, target)
    pred_f, tgt_f = pred[mask_free], target[mask_free]
    return F.mse_loss(pred_f, tgt_f)

def loss_bc_zero_disp(pred_norm, dynamic_features, bc_mask,
                      y_scaler, lam_bc: float = 1e-3):
    """
    Penaliza desplazamiento != 0 EN ESPACIO FÍSICO en nodos fijos (bc_mask=True).
    pred_norm: y_hat normalizado
    y_scaler: (mean, std) usados para normalizar y. Si None, asumimos ya absoluto.
    """
    # TODO: Include custom weights for each features(some could be noisier)
    if lam_bc <= 0 or bc_mask is None or bc_mask.sum() == 0:
        return pred_norm.new_tensor(0.0)
    if y_scaler is None:
        pred_phys = pred_norm
    else:
        ym, ys = y_scaler
        pred_phys = pred_norm * ys.to(pred_norm) + ym.to(pred_norm)
    dynamic_features = pred_phys[:, dynamic_features]
    return lam_bc * (dynamic_features[bc_mask] ** 2).mean()

def train_epoch(model, loader, opt, dyn_features_idx_y, x_scaler, y_scaler, delta_scaler, device='cuda', lam_smooth=1e-3, scaler=None, max_grad_norm=1.0):
    model.train()
    total, nodes = 0.0, 0

    xm, xs = x_scaler
    ym, ys = y_scaler
    for g in tqdm(loader, leave=False):
        g = g.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            delta_norm = model(g.x, g.edge_index, getattr(g, 'edge_attr', None))

            # delta_phys = denorm(delta_norm, ym, ys)
            delta_phys = denorm_with_scaler(delta_norm, delta_scaler)
            x_phys = denorm(g.x, xm, xs)
            y_real_phys = denorm(g.y, ym, ys)

            dyn_idx_x = torch.as_tensor(dyn_features_idx_y, device=device)
            y_hat_phys = x_phys[:, dyn_idx_x] + delta_phys



            bc_mask = getattr(g, 'bc_mask', None)
            if bc_mask is not None:
              bc_mask = bc_mask.bool()
              mask_free = ~bc_mask
            else:
              mask_free = None
            rigid_mask = getattr(g, 'rigid_mask', None)

            loss = smooth_edge_penalty(y_hat_phys, y_real_phys, g.edge_index, lam_smooth, bc_mask, rigid_mask)
            loss += masked_mse(y_hat_phys, y_real_phys, mask_free)
            loss += loss_bc_zero_disp(y_hat_phys, dyn_features_idx_y, bc_mask, None, LAM_BC)
        if scaler is not None:
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()
        total += loss.item() * g.num_nodes; nodes += g.num_nodes
    return total / max(nodes, 1)

@torch.no_grad()
def eval_epoch(model, loader, dyn_features_idx_y, x_scaler, y_scaler, delta_scaler, device='cuda', lam_smooth=1e-3):
    model.eval()
    total, nodes = 0.0, 0
    xm, xs = x_scaler
    ym, ys = y_scaler
    for g in tqdm(loader, leave=False):
        g = g.to(device)

        delta_norm = model(g.x, g.edge_index, getattr(g, 'edge_attr', None))

        # delta_phys   = denorm(delta_norm, ym, ys)
        delta_phys = denorm_with_scaler(delta_norm, delta_scaler)
        x_phys       = denorm(g.x, xm, xs)
        y_real_phys  = denorm(g.y, ym, ys)

        dyn_idx_x = torch.as_tensor(dyn_features_idx_y, device=device)
        y_hat_phys = x_phys[:, dyn_idx_x] + delta_phys

        bc_mask = getattr(g, 'bc_mask', None)
        if bc_mask is not None:
          bc_mask = bc_mask.bool()
          mask_free = ~bc_mask
        else:
          mask_free = None
        rigid_mask = getattr(g, 'rigid_mask', None)
        loss = smooth_edge_penalty(y_hat_phys, y_real_phys, g.edge_index, lam_smooth, bc_mask, rigid_mask)
        loss += masked_mse(y_hat_phys, y_real_phys, mask_free)
        loss += loss_bc_zero_disp(y_hat_phys, dyn_features_idx_y, bc_mask, None, LAM_BC)

        # Accumulators
        total += loss.item() * g.num_nodes
        nodes += g.num_nodes
    return total / max(nodes, 1)



In [165]:
# Prob. de teacher forcing (decae 1.0 -> 0.2 en 50 épocas, ajusta a gusto)
def p_teacher(epoch, p0=1.0, pmin=0.2, T=30):
  '''
  Con qué frecuencia usamos el ground truth en lugar de la prediccón.
  Si el rollout de validación explota,sube p-teacher(mayor pmin o mayor T o reduce K)
  Si el train baja muy lento, quizas p-teacher muy bajo
  '''
  return max(pmin, p0 - (p0 - pmin) * epoch / max(T, 1))

def train_epoch_k(
    model,
    train_static: dict,
    x_scaler,                # (xm, xs)  para normalizar x_t
    delta_scaler,            # (dm, ds)  para desnormalizar Δ
    dyn_idx_x,               # índices dinámicos en x (usa X_DYNAMIC_INDEX)
    edge_scaler=None,
    device='cuda',
    epoch=1,
    lam_smooth=1e-3,
    lam_bc=1e-2,
    k_min=1,
    k_max=8,
    scaler=None,
    opt=None,
    max_grad_norm=1.0,
):
    """
    Entrena con ventanas aleatorias de longitud K (teacher forcing programado).
    Trabaja SIEMPRE en espacio físico.
    """
    assert opt is not None, "Falta optimizer"
    model.train()

    xm, xs = x_scaler
    dm, ds = delta_scaler
    xm_d, xs_d = xm.to(device), xs.to(device)
    dm_d, ds_d = dm.to(device), ds.to(device)

    dyn_idx_x_t = torch.as_tensor(dyn_idx_x, device=device)
    disp_dims = torch.as_tensor([0, 1, 2], device=device)  # clamp y BC sólo en desplazamiento

    total_loss, total_nodes = 0.0, 0
    sim_ids = list(train_static.keys())
    random.shuffle(sim_ids)
    pbar = tqdm(sim_ids, desc=f"Train (epoch {epoch})", leave=False)

    for sid in pbar:
        info = train_static[sid]
        # Datos estáticos de la simulación (FÍSICOS, sin normalizar)
        edge_index = info['edge_index'].to(device)
        edge_attr  = info.get('edge_attr', None)
        if edge_attr is not None and edge_scaler is not None:
            edge_attr = (edge_attr - edge_scaler[0]) / edge_scaler[1]
        edge_attr  = edge_attr.to(device) if edge_attr is not None else None

        bc_mask    = info.get('bc_mask', None)
        if bc_mask is not None: bc_mask = bc_mask.to(device).bool()
        rigid_mask = info.get('rigid_mask', None)
        if rigid_mask is not None: rigid_mask = rigid_mask.to(device)

        y_real_phys = info['y_real'].to(device)  # (T-1, N, Ddyn) en físico
        x_t_phys    = info['x0'].to(device).clone()  # (N, Din) en físico
        Tm1, N, Ddyn = y_real_phys.shape

        if Tm1 < 1:
            pbar.set_postfix(skip="Tm1<1")
            continue

        # Selecciona ventana aleatoria [t0, t0+K)
        t0 = int(torch.randint(0, max(1, Tm1 - k_max + 1), (1,)).item())
        K  = int(torch.randint(k_min, min(k_max, Tm1 - t0) + 1, (1,)).item())

        if t0 > 0:
          # Estado en t0 para el bloque dinámico = y_real[t0-1]
          x_t_phys[:, dyn_idx_x_t] = y_real_phys[t0 - 1]

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            loss_accum = 0.0

            for k in range(K):
                # Normaliza entrada del paso (x_t)
                x_in_norm = (x_t_phys - xm_d) / xs_d

                # Predice Δ en normalizado y pásalo a físico con delta_scaler
                delta_norm = model(x_in_norm, edge_index, edge_attr)         # (N, Ddyn)
                delta_phys = delta_norm * ds_d + dm_d                         # (N, Ddyn)

                # Estado siguiente predicho (RESIDUAL)
                y_hat_phys = x_t_phys[:, dyn_idx_x_t] + delta_phys

                # Pérdidas (todo en físico)
                mask_free = None if bc_mask is None else ~bc_mask
                loss  = smooth_edge_penalty(y_hat_phys, y_real_phys[t0+k], edge_index, lam_smooth, bc_mask, rigid_mask)
                loss += masked_mse(y_hat_phys, y_real_phys[t0+k], mask_free)

                # BC = cero desplazamiento en nodos fijos
                if bc_mask is not None and bc_mask.any():
                    loss += (lam_bc * (y_hat_phys[bc_mask][:, disp_dims]**2).mean())

                loss_accum = loss_accum + loss

                # Scheduled sampling para el siguiente estado
                use_gt = (torch.rand(1, device=device).item() < p_teacher(epoch))
                x_next_dyn = y_real_phys[t0 + k] if use_gt else y_hat_phys.detach()

                x_t_phys = x_t_phys.clone()
                x_t_phys[:, dyn_idx_x_t] = x_next_dyn

            loss_final = loss_accum / float(K)

        # Backward
        if scaler is not None:
            scaler.scale(loss_final).backward()
            if max_grad_norm is not None:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(opt); scaler.update()
        else:
            loss_final.backward()
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()

        total_loss += loss_final.item() * N
        total_nodes += N

        avg = total_loss / max(total_nodes, 1)
        pbar.set_postfix(K=K, t0=t0, loss=f"{avg:.4f}")

    return total_loss / max(total_nodes, 1)

@torch.no_grad()
def eval_epoch_k(
    model, val_static, x_scaler, delta_scaler, dyn_idx_x,
    edge_scaler=None, device='cuda',
    lam_smooth=1e-3, lam_bc=1e-2, K_eval=8,
    teacher_forcing=True, sample_t0=False,
    use_tqdm=True,
):
    model.eval()

    xm, xs = x_scaler
    dm, ds = delta_scaler
    xm_d, xs_d = xm.to(device), xs.to(device)
    dm_d, ds_d = dm.to(device), ds.to(device)

    dyn_idx_x_t = torch.as_tensor(dyn_idx_x, device=device)
    disp_dims = torch.as_tensor([0, 1, 2], device=device)

    sim_ids = list(val_static.keys())
    total_loss, total_nodes = 0.0, 0
    pbar = tqdm(sim_ids, desc="Valid", leave=False) if use_tqdm else sim_ids

    for sid in pbar:
        info = val_static[sid]
        edge_index = info['edge_index'].to(device)
        edge_attr  = info.get('edge_attr', None)
        if edge_attr is not None and edge_scaler is not None:
            edge_attr = (edge_attr - edge_scaler[0]) / edge_scaler[1]
        edge_attr  = edge_attr.to(device) if edge_attr is not None else None

        bc_mask    = info.get('bc_mask', None)
        if bc_mask is not None: bc_mask = bc_mask.to(device).bool()
        rigid_mask = info.get('rigid_mask', None)
        if rigid_mask is not None: rigid_mask = rigid_mask.to(device)

        y_real_phys = info['y_real'].to(device)
        x_t_phys    = info['x0'].to(device).clone()
        Tm1, N, _   = y_real_phys.shape
        if Tm1 < 1:
            if use_tqdm: pbar.set_postfix(skip="Tm1<1")
            continue

        # t0 opcional para cubrir toda la secuencia
        if sample_t0 and Tm1 > K_eval:
            t0 = int(torch.randint(0, Tm1 - K_eval + 1, (1,)).item())
        else:
            t0 = 0

        K = min(K_eval, Tm1 - t0)

        # sembrar estado coherente con t0
        if t0 > 0:
            x_t_phys[:, dyn_idx_x_t] = y_real_phys[t0 - 1]

        loss_accum = 0.0
        for k in range(K):
            x_in_norm  = (x_t_phys - xm_d) / xs_d
            delta_norm = model(x_in_norm, edge_index, edge_attr)
            delta_phys = delta_norm * ds_d + dm_d
            y_hat_phys = x_t_phys[:, dyn_idx_x_t] + delta_phys

            mask_free = None if bc_mask is None else ~bc_mask
            loss  = smooth_edge_penalty(y_hat_phys, y_real_phys[t0+k], edge_index, lam_smooth, bc_mask, rigid_mask)
            loss += masked_mse(y_hat_phys, y_real_phys[t0+k], mask_free)
            if bc_mask is not None and bc_mask.any():
                loss += lam_bc * (y_hat_phys[bc_mask][:, disp_dims]**2).mean()
            loss_accum += loss

            # avance del estado
            if teacher_forcing:
                x_t_phys[:, dyn_idx_x_t] = y_real_phys[t0 + k]
            else:
                x_t_phys[:, dyn_idx_x_t] = y_hat_phys

        loss_mean = (loss_accum / float(K)).item()
        total_loss  += loss_mean * N
        total_nodes += N

        if use_tqdm:
            avg = total_loss / max(total_nodes, 1)
            pbar.set_postfix(K=K, t0=t0, val=f"{avg:.4f}")

    return total_loss / max(total_nodes, 1)

In [166]:
def save_checkpoint(path, model, x_scaler, y_scaler, delta_scaler, edge_scaler):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model_state": model.state_dict(),
                "x_scaler": x_scaler, "y_scaler": y_scaler,"delta_scaler": delta_scaler, "edge_scaler": edge_scaler}, path)
    print("Saved best checkpoint ->", path)

In [167]:
import time, torch, torch.optim as optim
from collections import defaultdict

start = time.time()
print("Building model...")

edge_dim = train_graphs[0].edge_attr.size(1) if hasattr(train_graphs[0], "edge_attr") and train_graphs[0].edge_attr is not None else 0
assert edge_dim > 0, "edge_attr required for GINEConv; si no tienes, cambia a un modelo sin edge_attr."

model = ImpactGNN_Edge(in_ch=INPUT_FEATURES, edge_attr_dim=edge_dim,
                       hidden=HIDDEN, out_ch=OUTPUT_FEATURES, layers=N_LAYERS, dropout=0.1).to(device)

opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

checkpoint_path = "/content/best_impact_gnn.pt"

best_val = float('inf')
best_state = None
wait = 15

print(f"Training for up to {MAX_EPOCHS} epochs...")

for epoch in range(1, MAX_EPOCHS+1):
    tr = train_epoch_k(model, train_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX, edge_scaler, device, epoch, LAM_SMOOTH, LAM_BC, 1, 8, scaler, opt, 1.0)
    # tr = train_epoch(model, train_loader, opt, Y_DYNAMIC_INDEX, x_scaler, y_scaler, delta_scaler,  device='cuda', lam_smooth=LAM_SMOOTH, scaler=scaler, max_grad_norm=1.0)
    # vl = eval_epoch(model, val_loader, Y_DYNAMIC_INDEX, x_scaler, y_scaler, delta_scaler, device='cuda', lam_smooth=LAM_SMOOTH)
    vl = eval_epoch_k(model, val_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX, edge_scaler, device, LAM_SMOOTH, LAM_BC, 8)
    vl_free = eval_epoch_k(model, val_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX, edge_scaler, device, LAM_SMOOTH, LAM_BC, 8, True, True)
    print(f"[Epoch {epoch:03d}] train {tr:.6f} | val {vl:.6f} | val_free {vl_free:.6f}")

    if vl + 1e-6 < best_val:
        wait = 0
        best_val = vl
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
        save_checkpoint(checkpoint_path, model, x_scaler, y_scaler, delta_scaler, edge_scaler)
    else:
        wait += 1
        if wait >= PATIENCE:
          print(f"Early stopping after {PATIENCE} epochs without improvement")
          break
end = time.time()
print(f"Elapsed time for training: {end - start:.1f}s.")

Building model...
Training for up to 200 epochs...


Train (epoch 1):   0%|          | 0/32 [00:00<?, ?it/s]

Valid:   0%|          | 0/7 [00:00<?, ?it/s]

Valid:   0%|          | 0/7 [00:00<?, ?it/s]

[Epoch 001] train 6078.427745 | val 107131.684660 | val_free 1556.154171
Saved best checkpoint -> /content/best_impact_gnn.pt


...

[Epoch 015] train 2506.347863 | val 111004.554885 | val_free 620.028669


Train (epoch 16):   0%|          | 0/32 [00:00<?, ?it/s]

Valid:   0%|          | 0/7 [00:00<?, ?it/s]

Valid:   0%|          | 0/7 [00:00<?, ?it/s]

[Epoch 016] train 8584.478861 | val 108249.174343 | val_free 23074.104494
Early stopping after 15 epochs without improvement
Elapsed time for training: 37.5s.


# Rollout

In [168]:
@torch.no_grad()
def rollout(
    model,
    T_eff: int,
    x0: torch.Tensor,                 # (N, Din) EN ESPACIO FÍSICO (desnormalizado)
    dyn_idx_x,                          # lista/LongTensor de índices dinámicos en x
    edge_index,
    edge_attr,
    x_scaler,                         # (x_mean, x_std) de Din cols
    y_scaler,                         # (y_mean, y_std) de Dout cols (las dinámicas que predices)
    delta_scaler,
    bc_mask=None,                     # Bool [N] (opcional)
    clamp_bc=False,                   # si quieres forzar 0 en desplazamientos de nodos fijos
    device='cuda',
    return_full=False                 # True -> devuelve la secuencia de x_t completas; False -> sólo y_hat por paso
):
    """
    x0: estado inicial con TODAS las columnas de entrada del modelo (Din).
        Si alguna estática no la tienes en x0, añádela antes.
    dyn_idx: posiciones en x que el modelo predice y que se actualizan en cada paso.
    disp_idx_in_dyn_x: subset dentro de las dinámicas que corresponde a desplazamientos.
    """
    xm, xs = x_scaler
    ym, ys = y_scaler

    xm_d, xs_d = xm.to(device), xs.to(device)
    ym_d, ys_d = ym.to(device), ys.to(device)

    dm, ds = delta_scaler  # NUEVO
    dm_d, ds_d = dm.to(device), ds.to(device)  # NUEVO

    x_t = x0.to(device)                              # (N, Din)


    # print(f"Initial : x_t{x_t[:10,:]}")
    edge_index = edge_index.to(device)
    edge_attr  = edge_attr.to(device) if edge_attr is not None else None

    if bc_mask is not None:
        bc_mask = bc_mask.to(device).bool()

    preds = []
    states = [x_t.clone()]

    for i in range(T_eff - 1):

        # normalize the input
        x_in = (x_t - xm_d) / xs_d         # (N, Din)

        # normalized output
        delta_norm  = model(x_in, edge_index, edge_attr)      # (N, Dout)
        # delta_phys = delta_norm * ys_d + ym_d   # (N, Dout)
        delta_phys = delta_norm * ds.to(delta_norm) + dm.to(delta_norm)

        if isinstance(dyn_idx_x, (list, tuple)):
            dyn_idx_x_t = torch.as_tensor(dyn_idx_x, device=device)
        else:
            dyn_idx_x_t = dyn_idx_x

        y_next_phys = x_t[:, dyn_idx_x_t] + delta_phys   # (N, D_out)

        # clamp a 0 en desplazamientos de nodos fijos (si aplica)
        # TODO: Implement
        if clamp_bc and bc_mask is not None and dyn_idx_x is not None:
            if isinstance(dyn_idx_x, (list, tuple)):
                dyn_idx_x = torch.as_tensor(dyn_idx_x, device=device)
            # y_hat[bc_mask, disp_idx_in_dyn] = 0
            # como y_hat es (N,Dout), indexa filas y columnas:
            y_next_phys[bc_mask, 0:3] = 0.0   # asumiendo [dx, dy, dz] en 0..2

        # actualiza x_t SOLO en las columnas dinámicas
        x_t = x_t.clone()
        x_t[:, dyn_idx_x_t] = y_next_phys

        preds.append(y_next_phys)
        if return_full:
            states.append(x_t.clone())

    return (torch.stack(states, 0) if return_full else torch.stack(preds, 0))

# Evaluate results

In [169]:
@torch.no_grad()
def compute_metrics(y_pred: torch.Tensor, y_real: torch.Tensor) -> Dict[str, float]:
    # Only compare the first 3 dimensions (displacements)
    diff = y_pred - y_real
    mae = (diff).abs().mean().item()
    rmse = torch.sqrt(((diff) ** 2).mean()).item()
    l2   = torch.norm(diff, dim=-1)  # [N]
    ade  = l2.mean().item()
    fde = (y_pred[-1] - y_real[-1]).abs().mean().item()
    return {'MAE': mae, 'RMSE': rmse, 'ADE': ade, 'FDE': fde}


In [170]:
if best_state is not None: model.load_state_dict(best_state)

print("Evaluating rollout on TEST simulations…")
model.eval()
metrics_all = []
for sid, info in test_static.items():
    edge_index = info['edge_index']
    edge_attr  = transform_edge_attr(info.get('edge_attr', None), edge_scaler)
    T_eff = info['T_eff']
    y_real = info['y_real'].to(device)
    x0 = info['x0']
    bc_mask = info['bc_mask']

    pred = rollout(model, T_eff, x0, X_DYNAMIC_INDEX, edge_index, edge_attr, x_scaler, y_scaler, delta_scaler, bc_mask, CLAMP_BC_IN_ROLLOUT, device, False)
    m = compute_metrics(pred, y_real)
    metrics_all.append(m)
    print(f"[SIM {sid}] MAE={m['MAE']:.6f} RMSE={m['RMSE']:.6f} ADE={m['ADE']:.6f} FDE={m['FDE']:.6f}")


if metrics_all:
    avg = {k: float(np.mean([d[k] for d in metrics_all])) for k in metrics_all[0].keys()}
    print("==== TEST AVERAGE ===="); [print(f"{k}: {v:.6f}") for k,v in avg.items()]

Evaluating rollout on TEST simulations…
[SIM 14] MAE=258.604584 RMSE=583.417236 ADE=1113.310791 FDE=220.460770
[SIM 45] MAE=251.290283 RMSE=573.843140 ADE=1090.825562 FDE=207.935730
[SIM 2] MAE=251.620575 RMSE=578.005981 ADE=1097.141602 FDE=211.581497
[SIM 36] MAE=253.158295 RMSE=575.560852 ADE=1092.531128 FDE=211.690826
[SIM 33] MAE=256.097229 RMSE=582.669067 ADE=1107.309204 FDE=220.987442
[SIM 1] MAE=255.410721 RMSE=577.274414 ADE=1106.875610 FDE=229.416473
[SIM 13] MAE=257.731598 RMSE=579.996887 ADE=1106.258545 FDE=227.335815
[SIM 8] MAE=256.049438 RMSE=579.777344 ADE=1103.598267 FDE=223.848877
==== TEST AVERAGE ====
MAE: 254.995340
RMSE: 578.818115
ADE: 1102.231339
FDE: 219.157179


# Results analysis

# Report:

---
---
## 28/02/2025

---
### Experimento 1
Experimento sólo con las Δx, Δy, Δz, Vx, Vy, Vz de la iteración número 3 de la BBDD.

**Añadimos los atributos de las velocidades!**

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-2, LAM_SMOOTH = 1e-3

Test Averages:

MAE: nan

RMSE: nan

ADE: nan

FDE: nan

---
### Experimento 2
Experimento sólo con las Δx, Δy, Δz, Vx, Vy, Vz de la iteración número 3 de la BBDD.

**Predecimos atributos medios!**

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-2, LAM_SMOOTH = 1e-3

Test Averages:

MAE: 12276151168.000000

RMSE: 289830109184.000000

ADE: 69336709120.000000

FDE: 382559735808.000000

**Video explota**

---
### Experimento 3
Experimento sólo con las Δx, Δy, Δz, Vx, Vy, Vz de la iteración número 3 de la BBDD.

**Incluimos scaler especifico para el Delta y hacemos arreglos en rollout y animacion_vtk!**

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-2, LAM_SMOOTH = 1e-3

Test Averages:

MAE: 905.745705

RMSE: 3255.191742

ADE: 4011.377686

FDE: 2235.110962

**Video explota pero al principio va muuucho mejor**

---
### Experimento 4
Experimento sólo con las Δx, Δy, Δz, Vx, Vy, Vz de la iteración número 3 de la BBDD.

**Incluimos K-step + scheduled sampling**

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-2, LAM_SMOOTH = 1e-3
K_MIN = 1, K_MAX = 8, p0=1.0, pmin=0.2, T(del profe)=30

Test Averages:

MAE: 323.336552

RMSE: 759.901100

ADE: 1446.807785

FDE: 347.623501

**Muucho mejor los videos**

---
### Experimento 5
Experimento sólo con las Δx, Δy, Δz, Vx, Vy, Vz de la iteración número 3 de la BBDD.

**Reducimos el MIN_T a 1**

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-2, LAM_SMOOTH = 1e-3
K_MIN = 1, K_MAX = 8, p0=1.0, pmin=0.2, T(del profe)=30

Test Averages:

MAE: 254.995340

RMSE: 578.818115

ADE: 1102.231339

FDE: 219.157179

**Muucho mejor los videos**

3D Animation

In [171]:
# pip install imageio imageio-ffmpeg

import numpy as np
import torch
import pyvista as pv
import imageio
from tqdm import trange

def unique_undirected_edges(edge_index: torch.Tensor):
    ei = edge_index.detach().cpu().numpy().T
    undirected = set()
    for u, v in ei:
        if u == v: continue
        a, b = (u, v) if u < v else (v, u)
        undirected.add((a, b))
    return np.array(list(undirected), dtype=np.int64)

def _build_vtk_lines(edges_uv: np.ndarray) -> np.ndarray:
    e = edges_uv.astype(np.int64, copy=False)
    counts = np.full((e.shape[0], 1), 2, dtype=np.int64)
    return np.hstack([counts, e]).ravel()

def animate_simulation_vtk(
    sim_info: dict,
    model=None, x_scaler=None, y_scaler=None, delta_scaler=None, dyn_idx_x=None,
    edge_scaler=None, device="cuda",
    save_path="rollout_vtk.mp4",
    max_edges=4000, stride=1, framerate=15,
    window_size=(1280, 720),
    show_points=True, point_size=3.0,
    show_bc=True, bc_point_size=9.0,
    tube_lines=False, line_width=1.0,
    camera="iso",
):
    # -------- Datos --------
    pos0       = sim_info["pos0"]
    edge_index = sim_info["edge_index"]
    y_real     = sim_info["y_real"]
    x0         = sim_info["x0"]
    T_eff      = sim_info["T_eff"]

    bc_mask    = sim_info.get("bc_mask", None)
    fixed_idx  = sim_info.get("fixed_idx", None)
    if bc_mask is None and fixed_idx is not None:
        N = y_real.shape[1]
        bc_mask = torch.zeros(N, dtype=torch.bool, device=y_real.device)
        bc_mask[fixed_idx] = True

    edge_attr = sim_info.get("edge_attr", None)
    if edge_attr is not None and edge_scaler is not None:
        edge_attr = transform_edge_attr(edge_attr, edge_scaler)

    # Predicción (si no viene precomputada)
    if model is not None:
        model.eval()
        with torch.no_grad():
            pred = rollout(
                model, T_eff, x0, dyn_idx_x, edge_index, edge_attr,
                x_scaler, y_scaler, delta_scaler, bc_mask, CLAMP_BC_IN_ROLLOUT,
                device, False
            )
    else:
        pred = sim_info["y_pred"]

    # -------- A NumPy --------
    pos0_np = pos0.detach().cpu().numpy()
    gt_np   = y_real.detach().cpu().numpy()[..., :3]
    pr_np   = pred.detach().cpu().numpy()[..., :3]
    Tm1, N, _ = gt_np.shape

    # -------- Edges --------
    edges_uv = unique_undirected_edges(edge_index)
    edges_uv = edges_uv.detach().cpu().numpy() if isinstance(edges_uv, torch.Tensor) else edges_uv
    if max_edges is not None and len(edges_uv) > max_edges:
        rng = np.random.RandomState(0)
        edges_uv = edges_uv[rng.choice(len(edges_uv), size=max_edges, replace=False)]
    vtk_lines = _build_vtk_lines(edges_uv)

    # -------- Rango espacial (solo para cámara) --------
    all_gt = pos0_np[None, ...] + gt_np
    all_pr = pos0_np[None, ...] + pr_np
    xyz_min = np.minimum(all_gt.min(axis=(0, 1)), all_pr.min(axis=(0, 1)))
    xyz_max = np.maximum(all_gt.max(axis=(0, 1)), all_pr.max(axis=(0, 1)))
    pad = 0.05 * (xyz_max - xyz_min + 1e-12)
    xyz_min -= pad; xyz_max += pad
    center = (xyz_min + xyz_max) / 2.0

    # -------- Dos plotters off-screen (mismo tamaño) --------
    single_size = (max(1, window_size[0] // 2), window_size[1])
    pv.global_theme.window_size = single_size
    pv.global_theme.anti_aliasing = "ssaa"
    pv.global_theme.smooth_shading = False
    pv.global_theme.background = "white"

    # === GT ===
    pl_gt = pv.Plotter(off_screen=True, window_size=single_size)
    pl_gt.set_background("white")
    pl_gt.add_text("Ground Truth", font_size=14)
    mesh_gt = pv.PolyData(pos0_np + gt_np[0], lines=vtk_lines)
    pl_gt.add_mesh(
        mesh_gt, color="seagreen", line_width=line_width,
        style="wireframe", lighting=False, render_lines_as_tubes=tube_lines,
        opacity=0.9, smooth_shading=False
    )
    pts_gt = pts_gt_free = pts_gt_fix = None
    if show_points:
        if show_bc and bc_mask is not None:
            bc_np = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np
            pts_gt_free = pv.PolyData((pos0_np + gt_np[0])[free_np])
            pts_gt_fix  = pv.PolyData((pos0_np + gt_np[0])[bc_np])
            pl_gt.add_mesh(pts_gt_free, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="seagreen", opacity=0.9)
            pl_gt.add_mesh(pts_gt_fix,  style="points", point_size=bc_point_size,
                           render_points_as_spheres=True, color="gold", opacity=1.0)
        else:
            pts_gt = pv.PolyData(pos0_np + gt_np[0])
            pl_gt.add_mesh(pts_gt, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="seagreen", opacity=0.9)
    if camera == "iso":
        pl_gt.view_isometric()
    pl_gt.show_bounds(xtitle="x", ytitle="y", ztitle="z")
    pl_gt.show_axes()
    pl_gt.reset_camera()
    pl_gt.reset_camera_clipping_range()

    # === Pred ===
    pl_pr = pv.Plotter(off_screen=True, window_size=single_size)
    pl_pr.set_background("white")
    pl_pr.add_text("Prediction", font_size=14)
    mesh_pr = pv.PolyData(pos0_np + pr_np[0], lines=vtk_lines)
    pl_pr.add_mesh(
        mesh_pr, color="crimson", line_width=line_width,
        style="wireframe", lighting=False, render_lines_as_tubes=tube_lines,
        opacity=0.9, smooth_shading=False
    )
    pts_pr = pts_pr_free = pts_pr_fix = None
    if show_points:
        if show_bc and bc_mask is not None:
            bc_np = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np
            pts_pr_free = pv.PolyData((pos0_np + pr_np[0])[free_np])
            pts_pr_fix  = pv.PolyData((pos0_np + pr_np[0])[bc_np])
            pl_pr.add_mesh(pts_pr_free, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="crimson", opacity=0.9)
            pl_pr.add_mesh(pts_pr_fix,  style="points", point_size=bc_point_size,
                           render_points_as_spheres=True, color="gold", opacity=1.0)
        else:
            pts_pr = pv.PolyData(pos0_np + pr_np[0])
            pl_pr.add_mesh(pts_pr, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="crimson", opacity=0.9)
    if camera == "iso":
        pl_pr.view_isometric()
    pl_pr.show_bounds(xtitle="x", ytitle="y", ztitle="z")
    pl_pr.show_axes()
    pl_pr.reset_camera()
    pl_pr.reset_camera_clipping_range()

    # -------- Encoder --------
    writer = imageio.get_writer(save_path, fps=framerate, codec="libx264", quality=8)

    # Primer frame
    img_gt = pl_gt.screenshot(return_img=True)
    img_pr = pl_pr.screenshot(return_img=True)
    frame  = np.concatenate([img_gt, img_pr], axis=1)
    writer.append_data(frame)

    # -------- Loop --------
    for t in trange(1, Tm1, desc="Render VTK (SxS)", leave=False):
        if (t % stride) != 0:
            continue

        Pgt = pos0_np + gt_np[t]
        Ppr = pos0_np + pr_np[t]

        # Actualiza geometría in-place
        mesh_gt.points = Pgt
        mesh_pr.points = Ppr

        if show_points and show_bc and bc_mask is not None:
            bc_np = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np
            if pts_gt_free is not None:
                pts_gt_free.points = Pgt[free_np]
                pts_gt_fix.points  = Pgt[bc_np]
            if pts_pr_free is not None:
                pts_pr_free.points = Ppr[free_np]
                pts_pr_fix.points  = Ppr[bc_np]
        elif show_points:
            if pts_gt is not None: pts_gt.points = Pgt
            if pts_pr is not None: pts_pr.points = Ppr

        # Render y captura de cada plotter
        pl_gt.render(); img_gt = pl_gt.screenshot(return_img=True)
        pl_pr.render(); img_pr = pl_pr.screenshot(return_img=True)
        frame = np.concatenate([img_gt, img_pr], axis=1)
        writer.append_data(frame)

    writer.close()
    pl_gt.close(); pl_pr.close()
    return save_path


## Creating animation

In [172]:
print("Creating  test animation...")
sid = next(iter(test_static.keys()))
animation_name = "rollout_test_" + str(sid) + ".mp4"
animation_path = "/content/drive/MyDrive/CrashGeoNN/" + animation_name
animate_simulation_vtk(test_static[sid],model,x_scaler,y_scaler, delta_scaler,X_DYNAMIC_INDEX,edge_scaler=edge_scaler,device="cuda",save_path=animation_path,
    max_edges=4000,stride=1,framerate=15,window_size=(1280, 720),show_points=True,point_size=3.0,show_bc=True,bc_point_size=9.0,
    tube_lines=False,          # True = tubos 3D (más bonito, algo más lento)line_width=1.0,
    camera="iso")
print("Creating  train animation...")
sid = next(iter(train_static.keys()))
animation_name = "rollout_train_" + str(sid) + ".mp4"
animation_path = "/content/drive/MyDrive/CrashGeoNN/" + animation_name
animate_simulation_vtk(train_static[sid],model,x_scaler,y_scaler, delta_scaler,X_DYNAMIC_INDEX,edge_scaler=edge_scaler,device="cuda",save_path=animation_path,
    max_edges=4000,stride=1,framerate=15,window_size=(1280, 720),show_points=True,point_size=3.0,show_bc=True,bc_point_size=9.0,
    tube_lines=False,          # True = tubos 3D (más bonito, algo más lento)line_width=1.0,
    camera="iso")

Creating  test animation...


Creating  train animation...


'/content/drive/MyDrive/CrashGeoNN/rollout_train_46.mp4'